In [1]:
%load_ext autoreload
%autoreload 2

In [21]:
import tqdm
import sys
import os
import time

In [3]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [5]:
sys.path.append(".")

In [6]:
from utils.datasets import SimpleSet, BerlinSparqlBenchmark
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever import QleverDB
from utils.dbs.fuseki import FusekiDB
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-05-12 21:01:14,905 - INFO - Loading faiss with AVX512 support.
2026-05-12 21:01:14,906 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-12 21:01:14,906 - INFO - Loading faiss with AVX2 support.
2026-05-12 21:01:14,906 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-05-12 21:01:14,907 - INFO - Loading faiss.
2026-05-12 21:01:14,933 - INFO - Successfully loaded faiss.


In [7]:
powers = np.arange(0, 4)  # extend on a more powerful machine
sizes = 10**powers

In [8]:
datasets: dict[int, BerlinSparqlBenchmark] = {}
raw_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Running BDSDM generation for size {size}...")
    dataset = BerlinSparqlBenchmark(base_dir=Path(f"./data/bsbm_{power}"), n=size)
    dataset.setup()
    datasets[power] = dataset
    # raw_sizes[power] = dataset.get_triple_count()

2026-05-12 21:01:15,151 - INFO - BSBM dataset already exists in data/bsbm_0, skipping generation
2026-05-12 21:01:15,151 - INFO - BSBM dataset already exists in data/bsbm_1, skipping generation
2026-05-12 21:01:15,152 - INFO - BSBM dataset already exists in data/bsbm_2, skipping generation
2026-05-12 21:01:15,152 - INFO - BSBM dataset already exists in data/bsbm_3, skipping generation


Running BDSDM generation for size 1...
Running BDSDM generation for size 10...
Running BDSDM generation for size 100...
Running BDSDM generation for size 1000...


In [9]:
encoded_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Encoding dataset of size {size}...")
    dataset = datasets[power]
    encoded_sizes[power] = dataset.encode_streaming(encoding_model)
    #  dataset.get_triple_count(encoded=True)

Encoding dataset of size 1...
Encoded TTL file already exists at data/bsbm_0/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10...
Encoded TTL file already exists at data/bsbm_1/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100...
Encoded TTL file already exists at data/bsbm_2/dataset_encoded.nt, skipping encoding
Encoding dataset of size 1000...
Encoded TTL file already exists at data/bsbm_3/dataset_encoded.nt, skipping encoding


## BSBM queries

### Simple Use case: 
find 10 products with a specific encoded label


### Complex Use case: 
For a specific product find 10 other similar products via their product label. 


In [10]:
test_label = "house furniture storage container"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
test_tensor.to_literal().n3()

'"{\\"data\\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.07163400948047638, -0.04559392109513283, 0.044428009539842606, 0.0004388350062072277, -0.004044292028993368, 0.05173112079501152, 0.08436278998851776, -0.028667306527495384, -0.032494205981492996, 0.058949727565050125, -0.0051073553040623665, 0.09506019204854965, -0.029170090332627296, -0.07172352075576782, 0.036426812410354614, 0.01788988709449768, 0.07774496078491211, -0.01058284193277359, -0.02719942107796669, -0.014839782379567623, 0.004996407311409712, -0.05195301026105881, -0.01563340425491333, -0.0004883587826043367, 0.012572054751217365, -0.043

In [12]:
from utils.dbs.neo4j import Neo4JDB


base_bsbm_set = datasets[3]

db_neo4j = Neo4JDB(
    id="timing-neo4j",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
)


2026-05-12 21:01:15,889 - WARNING - Killing any existing process using port 7687 before starting the server


 3348321

7687/tcp:           


In [13]:
base_bsbm_set.data_dir.name

'bsbm_3'

In [14]:
import os

os.getcwd()

'/nfsd/gracedata2/kantz/Dense-Vector-KG/benchmarks'

In [15]:
db_neo4j.URI

'neo4j://localhost:7687'

In [17]:

with db_neo4j:
    print("Running embedded Cypher query..., PID:", db_neo4j.pid)
    emb_result = db_neo4j.query_auto(
        test_tensor,
        query_difficulty=QUERY_DIFFICULTY.EASY,
        query_type=QUERY_TYPE.CYPHER_EMBEDDED
    )
    # sleep(3000)
emb_result

2026-05-12 21:01:38,840 - INFO - Copied default neo4j.conf to scratch/bsbm/bsbm_3/db/timing-neo4j/conf/neo4j.conf
2026-05-12 21:01:38,841 - INFO - Setting initial database to timing-neo4jbsbm3 in scratch/bsbm/bsbm_3/db/timing-neo4j/conf/neo4j.conf
2026-05-12 21:01:38,841 - WARNING - Killing any existing process using port 7687 before starting the server


7687/tcp:           
2026-05-12 21:01:38,904 - INFO - Starting Neo4J server with config from scratch/bsbm/bsbm_3/db/timing-neo4j/conf/neo4j.conf
2026-05-12 21:01:38,904 - WARNING - Server is already running, stopping it first
2026-05-12 21:01:38,905 - INFO - Stopping server!
2026-05-12 21:01:38,955 - ERROR - Command failed with return code 1
2026-05-12 21:01:38,957 - ERROR - Unable to retrieve routing information
2026-05-12 21:01:38,957 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j/sparql)


 3350471

2026-05-12 21:01:39,959 - ERROR - Unable to retrieve routing information
2026-05-12 21:01:39,959 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j/sparql)
2026-05-12 21:01:40,961 - ERROR - Unable to retrieve routing information
2026-05-12 21:01:40,961 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j/sparql)
2026-05-12 21:01:41,963 - ERROR - Unable to retrieve routing information
2026-05-12 21:01:41,963 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j/sparql)
2026-05-12 21:01:42,965 - ERROR - Unable to retrieve routing information
2026-05-12 21:01:42,965 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j/sparql)
2026-05-12 21:01:43,967 - ERROR - Unable to retrieve routing information

Running embedded Cypher query..., PID: 3350796


2026-05-12 21:01:48,806 - INFO - Stopping server!
2026-05-12 21:01:48,865 - ERROR - Command failed with return code 1


,product_id,score
0,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:18588,0.431962
1,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:4763,0.417647
2,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:18583,0.412833
3,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:11363,0.407595
4,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:18667,0.398600
5,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:10857,0.397672
6,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:8512,0.394696
7,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:8512,0.394696
8,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:13591,0.391172
9,4:03f72fbb-cb4f-445d-9c02-4e1e0e705863:9216,0.390203


In [17]:
# with db_neo4j:
#     index_result = db_neo4j.query_auto(
#         test_tensor,
#         query_difficulty=QUERY_DIFFICULTY.HARD,
#         query_type=QUERY_TYPE.CYPHER_EMBEDDED
#     )
#     # sleep(3000)
# index_result

In [18]:
import sys
sys.path.append(".")

In [25]:
from utils.datasets.dbpedia import DBPedia
from utils.dbs.neo4j import Neo4JDB
from pathlib import Path
dataset = DBPedia(base_dir=Path("data/dbpedia"))
db_neo4j_dbpedia = Neo4JDB(
    id="timing-neo4j-dbpedia",
    base_dir=Path(f"./scratch/dbpedia/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    force_recreate=True,
)
dataset.get_encoded_ttl_file().exists()

2026-05-12 21:15:02,222 - WARNING - Killing any existing process using port 7687 before starting the server
2026-05-12 21:15:02,269 - ERROR - Command failed with return code 1


True

In [26]:
with db_neo4j_dbpedia:
    emb_result = db_neo4j_dbpedia.query_auto(
        test_tensor,
        query_difficulty=QUERY_DIFFICULTY.EASY,
        query_type=QUERY_TYPE.CYPHER_EMBEDDED
    )
    
    time.sleep(3000)
emb_result

2026-05-12 21:15:03,221 - INFO - Copied default neo4j.conf to scratch/dbpedia/dbpedia/db/timing-neo4j-dbpedia/conf/neo4j.conf
2026-05-12 21:15:03,221 - INFO - Setting initial database to timing-neo4j-dbpediadbpedia in scratch/dbpedia/dbpedia/db/timing-neo4j-dbpedia/conf/neo4j.conf
2026-05-12 21:15:03,222 - WARNING - Killing any existing process using port 7687 before starting the server
2026-05-12 21:15:03,269 - ERROR - Command failed with return code 1
2026-05-12 21:15:03,269 - INFO - Starting Neo4J server with config from scratch/dbpedia/dbpedia/db/timing-neo4j-dbpedia/conf/neo4j.conf
/nfsd/gracedata2/kantz/Dense-Vector-KG/.venv/lib/python3.13/site-packages/neo4j/_sync/driver.py:1059: PreviewWarning: Passing key-word arguments to verify_connectivity() is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  preview_warn(
2026-05-12 21:15:03,271 - ERROR - Unable to retrieve routin

2026-05-12 21:15:04,273 - ERROR - Unable to retrieve routing information
2026-05-12 21:15:04,274 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j-dbpedia/sparql)
2026-05-12 21:15:05,275 - ERROR - Unable to retrieve routing information
2026-05-12 21:15:05,276 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j-dbpedia/sparql)
2026-05-12 21:15:06,278 - ERROR - Unable to retrieve routing information
2026-05-12 21:15:06,278 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j-dbpedia/sparql)
2026-05-12 21:15:07,280 - ERROR - Unable to retrieve routing information
2026-05-12 21:15:07,280 - INFO - Waiting for server to start..., got error: Unable to retrieve routing information (http://localhost:7687/timing-neo4j-dbpedia/sparql)
2026-05-12 21:15:08,282 - ERROR - Unable

ServiceUnavailable: Unable to retrieve routing information